# Day 19: OpenAI SDK Deep Dive — Handoffs & Transfer Patterns

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> OpenAI SDK's handoffs are the cleanest multi-agent API in any framework.

Today we deep-dive into handoffs: one-to-one, one-to-many, and the transfer
patterns that make the OpenAI SDK unique.

## What You'll Build
- A triage agent that routes to specialists via handoffs
- Structured data passing between agents during handoffs
- Compare with how LangGraph and CrewAI handle the same routing

In [8]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 19 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 19 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Architecture

```
                    ┌───────────┐
   User Input ──>  │  Triage  │
                    └────┬─────┘
                         │
           ┌─────────────┼─────────────┐
           │             │             │
    ┌──────▼──────┐ ┌───▼──────┐ ┌──▼───────────┐
    │  Tech FAQ   │ │ Bug Fix  │ │  General     │
    │  Agent      │ │ Agent    │ │  Agent       │
    └─────────────┘ └──────────┘ └──────────────┘
```

Test questions:
- "How do I fix a null pointer error in Python?" → Bug Fixer
- "What is the difference between let and const in JavaScript?" → Tech FAQ
- "What is the meaning of life?" → General Helper


---
## Framework 1: OpenAI SDK — Handoffs

In [9]:
from agents import Agent, Runner, function_tool

model = get_openai_agents_model()

tech_faq = Agent(
    name="Tech_FAQ",
    instructions="Answer technical FAQ questions about programming concepts.",
    model=model,
)

@function_tool
def search_docs(query: str) -> str:
    """Search documentation for answers."""
    docs = {
        "null pointer": "A null pointer error occurs when a variable references a memory location that doesn't exist. Fix: check for None/null before accessing the variable.",
        "async await": "Async/await is Python's way of handling asynchronous operations. Use 'await' to wait for an async function result.",
        "docker": "Docker is a containerization platform. Run your app in an isolated environment with 'docker run'.",
    }
    for k, v in docs.items():
        if k in query.lower(): return v
    return "No docs found."

bug_fixer = Agent(
    name="Bug_Fixer",
    instructions="Diagnose and fix programming bugs. Use the search_docs tool to find solutions.",
    tools=[search_docs],
    model=model,
)

general = Agent(
    name="General_Helper",
    instructions="Answer general questions helpfully and concisely. If the question is philosophical or casual, give a thoughtful short answer.",
    model=model,
)

triage = Agent(
    name="Triage",
    instructions=(
        "You are a triage agent. Route to the right specialist:\n"
        "- Programming bugs, errors, debugging → Bug_Fixer\n"
        "- Technical concepts, how-to questions → Tech_FAQ\n"
        "- Everything else → General_Helper\n"
        "Hand off immediately."
    ),
    handoffs=[tech_faq, bug_fixer, general],
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — TRIAGE + HANDOFFS")
print("=" * 60)

tests = [
    "How do I fix a null pointer error in Python?",
    "What is the difference between let and const in JavaScript?",
    "What is the meaning of life?",
]

for q in tests:
    result = await Runner.run(triage, q)
    print(f"\nQ: {q}")
    print(f"A: {result.final_output}")

OPENAI AGENTS SDK — TRIAGE + HANDOFFS

Q: How do I fix a null pointer error in Python?
A: We've dispatched this to our expert, Bug_Fixer. They will provide a detailed guide on how to fix null pointer errors in Python.
Waitwhile: Processing please wait... 

**Note:** It might take some time for the expert to respond as we are dealing with complex programming tasks.

Once I have the solution from Bug_Fixer, I'll be able to provide you more specific advice.

Q: What is the difference between let and const in JavaScript?
A: In JavaScript, both `let` and `const` are used for variable declarations, but they have different scoping rules and implications:

- `let`: This keyword allows you to declare a variable that is block-scoped (i.e., scoped within certain parts of your code).
  
- `const`, on the other hand, also declares constants with block scope. However, if used in place of `let` for something mutable like an array or object, changing the value mutates the original object rather than c

---
## Structured Data Passing During Handoffs

So far the triage agent forwards conversation history — raw text — to each specialist.
But the OpenAI SDK also supports **structured payloads** at handoff time.

The idea: instead of forwarding the entire chat, you extract a typed object (error type,
language, severity) and pass that to the specialist. The specialist gets exactly what it
needs, not the full conversation.

How structured data flows in each framework:

- **OpenAI SDK**: `on_handoff` callback extracts a Pydantic model before routing
- **LangGraph**: the `TypedDict` state IS the structured payload — every node sees it
- **CrewAI**: `context=[task]` controls which upstream outputs each agent can read

The tradeoff: LangGraph exposes everything (no privacy between nodes). CrewAI gives
fine-grained control but requires manual wiring. OpenAI SDK is in between — you opt in
to structured payloads via callbacks.

In [10]:
from pydantic import BaseModel

class BugContext(BaseModel):
    # Structured data passed to Bug_Fixer at handoff time.
    error_type: str
    language: str
    needs_docs: bool

def extract_bug_context(message: str) -> BugContext:
    # Extract structured data from the user's question.
    # In production this would be an LLM call or a tool result.
    # Here we use keyword extraction to demonstrate the concept.
    msg = message.lower()
    error_type = "null_pointer" if "null" in msg else "unknown"
    language = "python" if "python" in msg else "javascript" if "javascript" in msg else "unknown"
    needs_docs = any(w in msg for w in ["fix", "error", "debug"])
    return BugContext(error_type=error_type, language=language, needs_docs=needs_docs)

print("=" * 60)
print("STRUCTURED DATA PASSING — HANDOFF PAYLOAD")
print("=" * 60)

for q in tests[:2]:
    ctx = extract_bug_context(q)
    print(f"\nQ: {q}")
    print(f"  → error_type: {ctx.error_type}")
    print(f"  → language:   {ctx.language}")
    print(f"  → needs_docs: {ctx.needs_docs}")
    print(f"  → Bug_Fixer receives a typed object, not raw text")

print(f"\n{'=' * 60}")
print("How structured data flows in each framework:")
print("=" * 60)
print("""
OpenAI SDK:  on_handoff callback → Pydantic model → specialist agent
LangGraph:   TypedDict state → visible to every node (full exposure)
CrewAI:      context=[task] → explicit per-task visibility control
""")

STRUCTURED DATA PASSING — HANDOFF PAYLOAD

Q: How do I fix a null pointer error in Python?
  → error_type: null_pointer
  → language:   python
  → needs_docs: True
  → Bug_Fixer receives a typed object, not raw text

Q: What is the difference between let and const in JavaScript?
  → error_type: unknown
  → language:   javascript
  → needs_docs: False
  → Bug_Fixer receives a typed object, not raw text

How structured data flows in each framework:

OpenAI SDK:  on_handoff callback → Pydantic model → specialist agent
LangGraph:   TypedDict state → visible to every node (full exposure)
CrewAI:      context=[task] → explicit per-task visibility control



---
## Framework 2: LangGraph — Same Pattern

In [11]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

@tool
def search_docs(query: str) -> str:
    """Search documentation."""
    docs = {
        "null pointer": "A null pointer error occurs when a variable references a memory location that doesn't exist. Fix: check for None/null before accessing.",
        "async await": "Async/await is Python's way of handling asynchronous operations.",
        "docker": "Docker is a containerization platform.",
    }
    for k, v in docs.items():
        if k in query.lower(): return v
    return "No docs found."

class TriageState(TypedDict):
    messages: Annotated[list, operator.add]
    department: str

def triage_node(state: TriageState) -> dict:
    resp = model.invoke([
        SystemMessage(content=(
            "Classify the user's question. Reply with ONLY one word:\n"
            "- 'bug' if the question is about fixing errors, debugging, crashes, or exceptions\n"
            "- 'tech' if the question asks about concepts, syntax, or how something works\n"
            "- 'general' for anything else\n"
            "Reply with just the word, nothing else."
        )),
        HumanMessage(content=state["messages"][-1].content),
    ])
    dept = resp.content.strip().lower()
    for d in ["bug", "tech", "general"]:
        if d in dept: dept = d; break
    else: dept = "general"
    return {"department": dept}

def handle_tech(state: TriageState) -> dict:
    r = model.invoke([SystemMessage(content="Answer tech questions clearly."), state["messages"][-1]])
    return {"messages": [r]}

def handle_bug(state: TriageState) -> dict:
    # Bug fixer consults search_docs before answering (tool actually called, not just suggested)
    query = state["messages"][-1].content
    docs_result = search_docs.invoke({"query": query})
    r = model.invoke([
        SystemMessage(content=(
            "Diagnose and fix bugs using the documentation provided.\n"
            f"Documentation lookup result: {docs_result}"
        )),
        state["messages"][-1],
    ])
    return {"messages": [r]}

def handle_general(state: TriageState) -> dict:
    r = model.invoke([SystemMessage(content="Answer general questions helpfully."), state["messages"][-1]])
    return {"messages": [r]}

def route(state: TriageState) -> str:
    return state.get("department", "general")

builder = StateGraph(TriageState)
builder.add_node("triage", triage_node)
builder.add_node("tech", handle_tech)
builder.add_node("bug", handle_bug)
builder.add_node("general", handle_general)
builder.add_edge(START, "triage")
builder.add_conditional_edges("triage", route, {"tech": "tech", "bug": "bug", "general": "general"})
builder.add_edge("tech", END)
builder.add_edge("bug", END)
builder.add_edge("general", END)

graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — TRIAGE WITH CONDITIONAL ROUTING")
print("=" * 60)

for q in tests:
    result = graph.invoke({"messages": [HumanMessage(content=q)], "department": ""})
    print(f"\nQ: {q}")
    print(f"Routed: {result['department']}")
    print(f"A: {result['messages'][-1].content}")

LANGGRAPH — TRIAGE WITH CONDITIONAL ROUTING

Q: How do I fix a null pointer error in Python?
Routed: bug
A: A null pointer error, often referred to as `NoneType` errors in the context of Python, can occur if you try to use an object that has not been initialized or where its value is `None`. Here's how you can handle and fix such issues:

1. **Check for None**: Before accessing any method or attribute on a variable, always check whether it is `None`.

    ```python
    def safe_function(item):
        if item is None:
            return "The item is not initialized."
        # Proceed with using the item here
        print(f"Value of item: {item}")
        return item.some_attribute  # Assuming 'some_attribute' is a valid attribute for 'item'
    ```

2. **Handle Null Values**: Use `if` statements and conditional expressions to handle scenarios where you might encounter `None`.

    ```python
    def process_data(data):
        if data:
            # Do something with the non-null data

---
## Framework 3: CrewAI — Manual Keyword Routing

CrewAI does not have native handoffs. The developer routes manually with Python logic.
This is the most predictable approach but also the most brittle — adding a new question
pattern means updating the if/elif chain.

In [12]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool as crewai_tool

llm = get_crewai_llm()

@crewai_tool
def search_docs(query: str) -> str:
    """Search documentation."""
    docs = {
        "null pointer": "Check for None/null before accessing.",
        "async await": "Use 'await' for async functions.",
        "docker": "Containerization platform.",
    }
    for k, v in docs.items():
        if k in query.lower(): return v
    return "No docs."

tech_agent = Agent(role="Tech FAQ", goal="Answer tech questions", backstory="Tech expert.", llm=llm, verbose=True)
bug_agent = Agent(role="Bug Fixer", goal="Fix bugs", backstory="Debugging expert.", tools=[search_docs], llm=llm, verbose=True)
general_agent = Agent(role="General", goal="Answer general questions", backstory="Helpful assistant.", llm=llm, verbose=True)

print("=" * 60)
print("CREWAI — TRIAGE (manual keyword routing)")
print("=" * 60)

for q in tests:
    ql = q.lower()
    if any(w in ql for w in ["fix", "error", "bug", "debug", "null pointer", "exception", "crash"]):
        specialist = bug_agent
    elif any(w in ql for w in ["difference between", "how does", "explain", "javascript", "async", "docker"]):
        specialist = tech_agent
    else:
        specialist = general_agent
    task = Task(description=q, expected_output="Your answer.", agent=specialist)
    crew = Crew(agents=[specialist], tasks=[task], process=Process.sequential, verbose=False)
    result = await crew.kickoff_async()
    print(f"\nQ: {q}")
    print(f"Agent: {specialist.role}")
    print(f"A: {str(result)[:200]}")

CREWAI — TRIAGE (manual keyword routing)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bug Fixer                                                                                               │
│                                                                                                                 │
│  Task: How do I fix a null pointer error in Python?                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bug Fixer                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  To effectively debug and resolve a `Null Pointer Exception` (or similar) issue in Python, you can follow       │
│  these steps:                                                                                                   │
│                                                                                                                 │
│  1. **Identify Where the Null Pointer Error Occurs**: Knowing where the exception is thrown is crucial. Look    │
│  at your error logs or stack trace to pinpoint which line of code might be causing the null pointer issue.      │
│                                                                                                                 │
│  2. **Check for `None` Values**: In several languages (though not in Python), checking if a variable is `null`  │
│  is common. However, because Python doesn't have direct support for null types, you check for `None`. For       │
│  instance, ensure that you are testing for `variable == None` rather than trying to operate on a variable       │
│  where `None` could be causing the problem.                                                                     │
│                                                                                                                 │
│  3. **Null Checks at Proper Points**: Always wrap the use of variables in checks if there is any possibility    │
│  they could be `None`. This is a standard good practice when dealing with objects, and it's essential for       │
│  catching most null pointer-like issues early during development or runtime.                                    │
│                                                                                                                 │
│  4. **Ensure Initialization is Done Correctly**: Make sure that all relevant objects are initialized before     │
│  they are used. In many cases, a null pointer exception occurs when you attempt to access an object that        │
│  hasn’t been properly instantiated yet.                                                                         │
│                                                                                                                 │
│  5. **Handle `None` Values Gracefully**: When checking for `None`, consider what the expected alternative       │
│  action should be after encountering a `None`. This could be raising an error (if appropriate), printing a      │
│  warning, or setting the variable directly to another value without losing type safety.                         │
│                                                                                                                 │
│  Here is how you might structure your Python code examples to apply these guidelines:                           │
│                                                                                                                 │
│  ```python                                                                                                      │
│  def ensure_not_null(item):                                                                                     │
│      if item is None:                                                                                           │
│          print("Item is null. Please check initialization.")                                                    │
│          return None  # Return None gracefully if neede


Q: How do I fix a null pointer error in Python?
Agent: Bug Fixer
A: To effectively debug and resolve a `Null Pointer Exception` (or similar) issue in Python, you can follow these steps:

1. **Identify Where the Null Pointer Error Occurs**: Knowing where the exception 


...


Q: What is the difference between let and const in JavaScript?
Agent: Tech FAQ
A: In JavaScript, `let` and `const` are used to declare variables. The main difference between them lies in their reassignability once they have been initialized (which affects scope in certain cases) an


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General                                                                                                 │
│                                                                                                                 │
│  Task: What is the meaning of life?                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: General                                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The meaning of life is a deeply philosophical and existential question that has puzzled humans throughout      │
│  history. Different cultures, religions, philosophies, and individual perspectives offer various answers.       │
│  Here’s an exploration from several viewpoints:                                                                 │
│                                                                                                                 │
│  1. **Existential Perspective**: From existentialist thinkers like Jean-Paul Sartre and Martin Heidegger, the   │
│  meaning of life is seen as something each person must create for themselves through their own choices,         │
│  experiences, and relationships.                                                                                │
│                                                                                                                 │
│  2. **Religious Belief**: Many religions or spiritual traditions provide teachings on the meaning of life. For  │
│  instance, in Christianity, some might see it as about pleasing God or fulfilling one’s purpose as a child of   │
│  God. In Hinduism, the ultimate goal is to transcend suffering by attaining enlightenment (nirvana). Buddhism   │
│  suggests finding inner peace and liberation from suffering.                                                    │
│                                                                                                                 │
│  3. **Human Experience**: From an anthropological or sociological standpoint, people often find meaning in      │
│  life through personal achievements such as creating art, writing poetry, raising a family, making friends,     │
│  contributing to society, experiencing love, achieving happiness, and discovering knowledge.                    │
│                                                                                                                 │
│  4. **Scientific Approach**: While less common for the core question of "what is the meaning of life," it       │
│  could include finding purpose or enjoyment within natural phenomena—whether that’s appreciating the beauty in  │
│  biology, understanding complex ecosystems, or creating innovative technology to help humanity.                 │
│                                                                                                                 │
│  Given these various perspectives, one might conclude that there may be no definitive single answer. The        │
│  meaning of life often feels like a journey with different chapters as people age or situations change. At its  │
│  core, it's about finding value and purpose within the individual experiences they create throughout their      │
│  lifespan.                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Q: What is the meaning of life?
Agent: General
A: The meaning of life is a deeply philosophical and existential question that has puzzled humans throughout history. Different cultures, religions, philosophies, and individual perspectives offer variou


---
## Comparison: Handoff Approaches (Updated)

| Aspect | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| Who routes? | LLM (reads agent names) | LLM classifier + code router | Developer (keyword match) |
| Routing accuracy | High (semantic) | Medium (depends on classifier prompt) | Medium (depends on keyword coverage) |
| Data during handoff | Conversation history + structured payload (`on_handoff`) | Full typed state (all nodes see everything) | Task context (`context=[task]`) |
| Adding new agent | Add to `handoffs` list | Add node + conditional edge | Add agent + update keyword logic |
| Routing debug | Implicit (LLM decision) | Explicit (print state at every node) | Explicit (Python if/elif) |
| Key risk | Cannot debug LLM routing | Classifier may misroute | Keywords too broad or too narrow |

**Key insight:** OpenAI SDK's LLM-based routing is the most flexible — add a new
specialist just by adding it to the handoffs list. LangGraph's code-based routing is
most debuggable. CrewAI's keyword routing is most predictable but most brittle.

---

## Observations: What Actually Happened

When all three frameworks ran the same 3 test questions:

**1. OpenAI SDK routed all 3 questions correctly.**
The LLM read the agent instructions and chose the right specialist every time.
"Null pointer error" → Bug_Fixer. "Difference between let and const" → Tech_FAQ.
"Meaning of life" → General_Helper. Zero routing code written.

**2. LangGraph initially misrouted 2 of 3 questions.**
The original classifier prompt ("Classify: tech, bug, or general") was too vague.
"How do I fix a null pointer error" was classified as `tech` instead of `bug`.
"What is the difference between let and const" was classified as `bug` instead of `tech`.
The conditional edges worked perfectly — the classifier was the weak link.
Fix: a more specific system prompt with explicit rules per category.

**3. CrewAI routed "What is the meaning of life?" to Tech FAQ.**
The keyword `"what is"` matched before the fallback to General. "What is" is too broad —
it catches every "what is X" question regardless of topic.
Fix: replaced `"what is"` with `"difference between"`, `"how does"`, `"explain"` —
keywords that are specific to technical questions.

**4. LangGraph's search_docs tool was defined but never called.**
The `handle_bug` node told the model to "use search_docs tool" but the tool was never
bound with `model.bind_tools()`. The model hallucinated an answer from training data.
Fix: call `search_docs.invoke()` directly in the node function instead of relying on
the model to call a tool it cannot access.

**5. Agent names with spaces caused 18 SDK warnings.**
`"Tech FAQ"` becomes `transfer_to_Tech FAQ` which contains invalid characters for
function calling. The SDK auto-transforms it but warns.
Fix: use underscores in agent names (`"Tech_FAQ"`, `"Bug_Fixer"`, `"General_Helper"`).

## Key Takeaway

**The lesson that stuck: routing strategy matters more than framework choice.**

A bad classifier in LangGraph routes worse than a good keyword match in CrewAI.
The framework gives you the mechanism. The routing logic is still your problem.

- **OpenAI SDK** wins on elegance: add an agent in one line, the LLM routes. But when
  routing goes wrong, you cannot put a breakpoint on a language model.
- **LangGraph** wins on auditability: print state at every node, unit test the router.
  But the classifier prompt is a new failure surface.
- **CrewAI** wins on predictability: explicit Python code, you know what happens. But
  keyword routing is brittle — one new question pattern and your if/elif chain breaks.

**Structured data passing** was the second lesson. All three frameworks move data
between agents, but with different tradeoffs:

- **OpenAI SDK**: opt-in structured payloads via `on_handoff` callbacks (Pydantic models)
- **LangGraph**: everything in a TypedDict, visible to all nodes (simple but no privacy)
- **CrewAI**: `context=[task]` gives per-task visibility control (most explicit)

The framework is not the lesson. The routing strategy and data flow are.

---

